# Phần 1: Tiền xử lý và kết nối 5 bảng dữ liệu

Đồ án cuối kỳ môn Tương tác Dữ liệu
Người phụ trách: Đinh Quốc Anh (24133003), Data Engineer & Data Analyst
Đề tài: Trực quan hóa đặc trưng âm thanh và dự đoán xu hướng bài hát thịnh hành trên Spotify

Notebook này làm các việc sau:

1. Nạp 5 bảng dữ liệu thô (khoảng 1.9 triệu bản ghi).
2. Làm sạch từng bảng (xử lý kiểu dữ liệu, missing, outliers, kẹp miền giá trị).
3. Chuẩn hóa tên quốc gia World Bank và kết nối 5 bảng thành file master 42 cột.
4. Xuất file dữ liệu master và dữ liệu cấp bài hát phục vụ Machine Learning.

Các file đầu ra:

- `data/processed/merged_spotify_data.csv`: file master, 1,078,125 dòng x 42 cột (~338.6 MB). Gồm đầy đủ dữ liệu kinh tế 74 nước và cột `covid_phase`.
- `data/processed/tracks_ml_data.csv`: 36,699 bài hát x 18 cột (~5.0 MB), mỗi bài một dòng để tránh rò rỉ dữ liệu (data leakage) khi train Logistic Regression.

## 0. Khởi tạo môi trường

In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# xác định thư mục gốc của dự án
NOTEBOOK_DIR = os.getcwd()
if os.path.basename(NOTEBOOK_DIR) == 'notebooks':
    PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f'Thư mục dự án: {PROJECT_ROOT}')
print(f'Pandas {pd.__version__}, NumPy {np.__version__}')
print('Chế độ chạy: toàn bộ dữ liệu (1.78 triệu dòng thô -> 1.08 triệu dòng master 2020-2022)')

Thư mục dự án: D:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics
Pandas 3.0.3, NumPy 2.4.6
Chế độ chạy: toàn bộ dữ liệu (1.78 triệu dòng thô -> 1.08 triệu dòng master 2020-2022)


In [2]:
from src.data_loader import (
    load_raw_data,
    clean_raw_charts,
    clean_raw_tracks,
    clean_raw_artists,
    clean_raw_gdp,
    clean_raw_country_codes,
    build_master_dataset,
    generate_ml_dataset,
    load_master_data,
    load_ml_sample,
    data_quality_report,
    RAW_DIR, PROCESSED_DIR,
    FINAL_INSIGHT_COLUMNS, AUDIO_FEATURES,
    COUNTRY_NAME_MAPPING,
)

## 1. Nạp dữ liệu thô

Năm bảng dữ liệu:

- `spotify-weekly-charts.csv`: bảng xếp hạng tuần của Spotify, 1,787,999 dòng x 36 cột (~797 MB).
- `spotify-tracks-dataset.csv`: 114,000 bài hát kèm đặc trưng âm học và thể loại (~20 MB).
- `gdp_data.csv`: GDP theo quốc gia và năm (World Bank), 13,365 dòng x 4 cột.
- `country_codes.csv`: khu vực và nhóm thu nhập của 217 quốc gia (World Bank).
- `best-selling-artists.csv`: 121 nghệ sĩ bán đĩa nhiều nhất lịch sử.

In [3]:
charts_raw, tracks_raw, gdp_raw, country_codes_raw, artists_raw = load_raw_data()

Nạp dữ liệu thô
  đang đọc spotify-weekly-charts.csv (~800 MB)...


  charts: 1,787,999 dòng, 36 cột
  đang đọc spotify-tracks-dataset.csv...


  tracks: 114,000 dòng, 22 cột
  gdp: 13,365 dòng, 4 cột
  country codes: 217 dòng, 3 cột
  best-selling artists: 121 dòng, 7 cột
  tổng: 1,915,702 bản ghi


### 1b. Khảo sát chất lượng 5 bảng thô

Xem kích thước, tỷ lệ missing và trùng lặp của từng bảng trước khi làm sạch.

In [4]:
# Bảng 1: weekly charts
print('Bảng 1: Spotify weekly charts')
print(f'Shape: {charts_raw.shape[0]:,} dòng x {charts_raw.shape[1]} cột')
print(f'Số track URI khác nhau: {charts_raw["uri"].nunique():,}')
print(f'Số quốc gia: {charts_raw["country"].nunique()}')

dup1 = charts_raw.duplicated(subset=['uri', 'week', 'country']).sum()
print(f'\nDòng trùng lặp: {dup1:,}')

missing1 = charts_raw.isnull().sum()
missing1_pct = (charts_raw.isnull().mean() * 100).round(2)
miss_df = pd.DataFrame({'Missing': missing1, '%': missing1_pct}).query('Missing > 0')
print(f'Số cột có giá trị thiếu: {len(miss_df)}')
miss_df

Bảng 1: Spotify weekly charts
Shape: 1,787,999 dòng x 36 cột
Số track URI khác nhau: 39,203
Số quốc gia: 75



Dòng trùng lặp: 678,478
Số cột có giá trị thiếu: 14


,Missing,%
artist_img,9319,0.52
album_cover,17,0.00
danceability,395,0.02
energy,395,0.02
key,395,0.02
mode,395,0.02
loudness,395,0.02
speechiness,395,0.02
acousticness,395,0.02
instrumentalness,395,0.02


In [5]:
# Bảng 2: tracks dataset
print('Bảng 2: Spotify tracks dataset')
print(f'Shape: {tracks_raw.shape[0]:,} dòng x {tracks_raw.shape[1]} cột')
print(f'track_id khác nhau: {tracks_raw["track_id"].nunique():,} / {len(tracks_raw):,}')
audio_chk = ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness']
tracks_raw[[c for c in audio_chk if c in tracks_raw.columns]].describe().round(3)

Bảng 2: Spotify tracks dataset
Shape: 114,000 dòng x 22 cột
track_id khác nhau: 89,741 / 114,000


,danceability,energy,valence,tempo,loudness,acousticness
count,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000
mean,0.567,0.641,0.474,122.148,-8.259,0.315
std,0.174,0.252,0.259,29.978,5.029,0.333
min,0.000,0.000,0.000,0.000,-49.531,0.000
25%,0.456,0.472,0.260,99.219,-10.013,0.017
50%,0.580,0.685,0.464,122.017,-7.004,0.169
75%,0.695,0.854,0.683,140.071,-5.003,0.598
max,0.985,1.000,0.995,243.372,4.532,0.996


In [6]:
# Bảng 3, 4, 5
print('Bảng 3: GDP (World Bank)')
print(f'Shape: {gdp_raw.shape} | Quốc gia: {gdp_raw["country_name"].nunique()} | Năm: {gdp_raw["year"].min()}-{gdp_raw["year"].max()}')
print(f'GDP missing: {gdp_raw["value"].isnull().sum():,} ({gdp_raw["value"].isnull().mean()*100:.1f}%)')

print('\nBảng 4: Country codes')
print(f'Shape: {country_codes_raw.shape} | Khu vực: {country_codes_raw["region"].nunique()} | Nhóm thu nhập: {country_codes_raw["income_group"].nunique()}')

print('\nBảng 5: Best-selling artists')
print(f'Shape: {artists_raw.shape}')
artists_raw[['Artist name', 'Country', 'Genre']].head(3)

Bảng 3: GDP (World Bank)
Shape: (13365, 4) | Quốc gia: 262 | Năm: 1960-2022
GDP missing: 0 (0.0%)

Bảng 4: Country codes
Shape: (217, 3) | Khu vực: 7 | Nhóm thu nhập: 4

Bảng 5: Best-selling artists
Shape: (121, 7)


,Artist name,Country,Genre
0,The Beatles,United Kingdom,Rock/pop
1,Michael Jackson,United States,Pop / rock /dance/soul/R&B
2,Elvis Presley,United States,Rock and roll/ pop /country


## 2. Làm sạch bảng charts

Các vấn đề trong dữ liệu thô và cách xử lý:

- `uri` có dạng `spotify:track:XXXX`: tách lấy `track_id` (22 ký tự).
- `week` là chuỗi: đổi sang datetime, lấy thêm `year`, `month`.
- `streams` có giá trị rỗng hoặc <= 0: ép sang số nguyên và bỏ các dòng <= 0.
- 9 audio features: ép sang float.
- Trùng lặp: bỏ trùng theo bộ ba `(track_id, week, country)`.
- Bỏ sớm các cột không dùng đến (`uri`, `artist_names`, `artists_num`, `artist_id`, `key`, `mode`, `source`, `album_num_tracks`, `collab`, `release_date`, `previous_rank`, `artist_img`, `album_cover`, `pivot`, `region`, ...).

In [7]:
charts_clean = clean_raw_charts(charts_raw)

print(f'\nShape sau làm sạch: {charts_clean.shape[0]:,} dòng x {charts_clean.shape[1]} cột')
print(f'track_id mẫu: {charts_clean["track_id"].iloc[0]}')
print(f'Kiểu dữ liệu của week: {charts_clean["week"].dtype}')
print(f'Năm: {charts_clean["year"].min()} - {charts_clean["year"].max()}')
print(f'Streams: {charts_clean["streams"].min():,} - {charts_clean["streams"].max():,}')
charts_clean[['track_id', 'track_name', 'artist_individual', 'country', 'week', 'rank', 'streams']].head(3)


Làm sạch bảng charts


  1,787,999 -> 1,109,520 dòng, 23 cột

Shape sau làm sạch: 1,109,520 dòng x 23 cột
track_id mẫu: 2gpQi3hbcUAcEG8m2dlgfB
Kiểu dữ liệu của week: datetime64[us]
Năm: 2016 - 2022
Streams: 2,067 - 84,952,932


,track_id,track_name,artist_individual,country,week,rank,streams
0,2gpQi3hbcUAcEG8m2dlgfB,Plan A,Paulo Londra,Argentina,2022-04-14,1,3003411
1,2x8oBuYaObjqHqgGuIUZ0b,ARRANCARMELO,WOS,Argentina,2022-04-14,2,2512175
2,2SJZdZ5DLtlRosJ2xHJJJa,Chance,Paulo Londra,Argentina,2022-04-14,3,2408983


## 3. Làm sạch bảng tracks

- `track_id` bị trùng: sắp theo `popularity` giảm dần, giữ bản có popularity cao nhất.
- 7 chỉ số dạng tỉ lệ được kẹp về khoảng [0, 1].
- `loudness` kẹp về [-60, 5] dB, `tempo` kẹp về [0, 300] BPM.
- Chỉ giữ các cột cần cho phép join: `track_id`, `track_genre`, `popularity`, `explicit`.

In [8]:
tracks_clean = clean_raw_tracks(tracks_raw)

print(f'\nShape: {tracks_clean.shape[0]:,} bài hát x {tracks_clean.shape[1]} cột')
print(f'Số thể loại: {tracks_clean["track_genre"].nunique()}')
print('Top 5 thể loại:')
print(tracks_clean['track_genre'].value_counts().head(5))

  tracks sau làm sạch: 89,741 bài hát

Shape: 89,741 bài hát x 4 cột
Số thể loại: 114
Top 5 thể loại:
track_genre
tango        1000
study         998
sleep         997
grindcore     995
idm           994
Name: count, dtype: int64


## 4. Làm sạch nghệ sĩ, GDP và country codes

- Best-selling artists: đưa tên nghệ sĩ về chữ thường, bỏ khoảng trắng thừa để khớp khi join.
- GDP: đổi tên `value` thành `gdp_value`, ép kiểu `year`.
- Country codes: đổi tên `region` thành `region_wb`, giữ `country_code`, `region_wb`, `income_group`; bổ sung mã chuẩn `TWN` (High income, East Asia & Pacific) cho Đài Loan.

In [9]:
artists_clean = clean_raw_artists(artists_raw)
gdp_clean = clean_raw_gdp(gdp_raw)
country_codes_clean = clean_raw_country_codes(country_codes_raw)

print(f'Artists: {len(artists_clean)} nghệ sĩ')
print(f'GDP: {len(gdp_clean):,} bản ghi ({gdp_clean["year"].min()}-{gdp_clean["year"].max()})')
print(f'Country codes: {len(country_codes_clean)} quốc gia/vùng lãnh thổ')

  best-selling artists: 121 nghệ sĩ
Artists: 121 nghệ sĩ
GDP: 13,365 bản ghi (1960-2022)
Country codes: 218 quốc gia/vùng lãnh thổ


## 5. Xây dựng master dataset

Bắt đầu từ bảng charts (lọc 2020-2022) và dùng 4 phép LEFT JOIN theo thứ tự:

| Bước | Bảng ghép vào | Khóa | Cột thêm vào |
|---|---|---|---|
| 1 | gdp_data | `country_code`, `year` (sau khi chuẩn hóa 7 quốc gia sang tên World Bank & gán mã ISO-3) | `gdp_value` |
| 2 | country_codes | `country_code` | `region_wb`, `income_group` |
| 3 | tracks_dataset | `track_id` | `track_genre`, `popularity`, `explicit` |
| 4 | best_selling | tên nghệ sĩ (đã chuẩn hóa) | `legacy_artist_name` |

Việc chuẩn hóa 7 quốc gia (`Korea`, `Turkey`, `Czech Republic`, `Hong Kong`, `Egypt`, `Slovakia`, `Venezuela`) và gán mã ISO-3 trực tiếp giúp tỷ lệ khớp GDP đạt **98.2%** (chỉ còn thiếu Taiwan do World Bank không báo cáo và Venezuela do thiếu số liệu 2020-2022), đồng thời giảm số dòng thiếu `income_group` và `region_wb` về **0**.

In [10]:
df_master = build_master_dataset(
    charts_clean, gdp_clean, country_codes_clean, tracks_clean, artists_clean, save=False
)

print(f'\nShape: {df_master.shape[0]:,} dòng x {df_master.shape[1]} cột')
print(f'Số quốc gia/vùng: {df_master["country"].nunique()}')
rate_gdp = df_master[df_master['country'] != 'Global']['gdp_value'].notna().mean() * 100
missing_inc = df_master[df_master['country'] != 'Global']['income_group'].isna().sum()
print(f'Tỷ lệ khớp GDP (bỏ Global): {rate_gdp:.1f}%')
print(f'Số dòng thiếu income_group/region_wb: {missing_inc:,}')
print(f'Tỷ lệ khớp thể loại (tracks): {df_master["track_genre"].notna().mean()*100:.1f}%')
print(f'Số dòng của nghệ sĩ huyền thoại: {df_master["is_legacy_artist"].sum():,}')


Xây dựng master dataset


  lọc 2020-2022: 1,078,125 dòng


  master: 1,078,125 dòng, 42 cột

Shape: 1,078,125 dòng x 42 cột
Số quốc gia/vùng: 74


Tỷ lệ khớp GDP (bỏ Global): 98.2%
Số dòng thiếu income_group/region_wb: 0
Tỷ lệ khớp thể loại (tracks): 33.8%
Số dòng của nghệ sĩ huyền thoại: 111,278


## 6. Kiểm tra các cột tính thêm

Có 13 cột được tính thêm từ dữ liệu gốc:

| # | Cột | Cách tính |
|---|---|---|
| 1 | `is_trending` | 1 nếu `rank` <= 50 |
| 2 | `is_popular` | 1 nếu `popularity` >= 70 |
| 3 | `mood_category` | chia 4 nhóm theo valence và energy (mô hình Russell) |
| 4 | `stream_share_country` | streams của bài / tổng streams của quốc gia trong tuần |
| 5 | `covid_phase` | 2020: Lockdown, 2021: Vaccine, 2022: Reopening |
| 6 | `energy_loudness_index` | (energy + loudness đã min-max) / 2 |
| 7 | `is_latin` | genre khớp từ khóa Latin, hoặc ngôn ngữ Spanish/Portuguese |
| 8 | `is_kpop` | genre khớp từ khóa K-Pop, hoặc ngôn ngữ Korean |
| 9 | `is_legacy_artist` | 1 nếu nghệ sĩ nằm trong danh sách bán đĩa nhiều nhất |
| 10 | `duration_min` | `duration` / 60,000 |
| 11 | `audio_diversity_score` | độ lệch chuẩn của danceability, energy, valence, acousticness |
| 12 | `log_streams` | ln(1 + streams) |
| 13 | `gdp_log` | ln(1 + gdp_value) |

In [11]:
calc_fields = [
    'is_trending', 'is_popular', 'mood_category', 'stream_share_country',
    'covid_phase', 'energy_loudness_index', 'is_latin', 'is_kpop',
    'is_legacy_artist', 'duration_min', 'audio_diversity_score',
    'log_streams', 'gdp_log',
]

for field in calc_fields:
    if field in df_master.columns:
        null_cnt = df_master[field].isnull().sum()
        uniq_cnt = df_master[field].nunique()
        print(f'{field:<24s} dtype={str(df_master[field].dtype):<10s} null={null_cnt:>6,} unique={uniq_cnt:>6,}')
    else:
        print(f'{field:<24s} không tìm thấy')

is_trending              dtype=int64      null=     0 unique=     2
is_popular               dtype=int64      null=     0 unique=     2
mood_category            dtype=str        null=     0 unique=     4
stream_share_country     dtype=float64    null=     0 unique=    86
covid_phase              dtype=str        null=     0 unique=     3
energy_loudness_index    dtype=float64    null=   145 unique=   699
is_latin                 dtype=int64      null=     0 unique=     2
is_kpop                  dtype=int64      null=     0 unique=     2
is_legacy_artist         dtype=int64      null=     0 unique=     2
duration_min             dtype=float64    null=   145 unique= 4,473
audio_diversity_score    dtype=float64    null=   145 unique=   449
log_streams              dtype=float64    null=     0 unique= 9,854
gdp_log                  dtype=float64    null=45,198 unique=   133


In [12]:
print('Phân phối mood_category:')
mood_dist = df_master['mood_category'].value_counts()
mood_pct = df_master['mood_category'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Số dòng': mood_dist, 'Tỷ lệ %': mood_pct.round(2)}))

print('\nPhân phối covid_phase:')
covid_dist = df_master['covid_phase'].value_counts()
covid_pct = df_master['covid_phase'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Số dòng': covid_dist, 'Tỷ lệ %': covid_pct.round(2)}))

print('\nLatin / K-Pop / nghệ sĩ huyền thoại:')
print(f'Latin : {df_master["is_latin"].sum():>8,} ({df_master["is_latin"].mean()*100:.1f}%)')
print(f'K-Pop : {df_master["is_kpop"].sum():>8,} ({df_master["is_kpop"].mean()*100:.1f}%)')
print(f'Legacy: {df_master["is_legacy_artist"].sum():>8,} ({df_master["is_legacy_artist"].mean()*100:.1f}%)')

Phân phối mood_category:
                   Số dòng  Tỷ lệ %
mood_category                      
Happy Energetic     540833    50.16
Tense Aggressive    330965    30.70
Sad Melancholic     146885    13.62
Peaceful Positive    59442     5.51

Phân phối covid_phase:


                        Số dòng  Tỷ lệ %
covid_phase                             
Vaccine & Escapism       662732    61.47
Reopening & New Normal   404793    37.55
Lockdown & Shock          10600     0.98

Latin / K-Pop / nghệ sĩ huyền thoại:
Latin :  311,219 (28.9%)
K-Pop :   64,867 (6.0%)
Legacy:  111,278 (10.3%)


## 7. Kiểm tra chất lượng bộ dữ liệu cuối

Kiểm tra lại master dataset trước khi xuất file.

In [13]:
print(f'Số dòng: {len(df_master):,}')
print(f'Số cột : {len(df_master.columns)}')

total_cells = len(df_master) * len(df_master.columns)
total_null = df_master.isnull().sum().sum()
print(f'Tổng số ô: {total_cells:,}')
print(f'Ô bị thiếu: {total_null:,} ({total_null/total_cells*100:.2f}%)')

print('\nDanh sách cột:')
for i, col in enumerate(df_master.columns, 1):
    print(f'{i:2d}. {col:<26s} ({df_master[col].dtype})')

Số dòng: 1,078,125
Số cột : 42
Tổng số ô: 45,281,250
Ô bị thiếu: 2,314,287 (5.11%)

Danh sách cột:
 1. track_id                   (str)
 2. track_name                 (str)
 3. artist_individual          (str)
 4. artist_genre               (str)
 5. country                    (str)
 6. language                   (str)
 7. week                       (datetime64[us])
 8. year                       (Int64)
 9. month                      (Int64)
10. rank                       (Int64)
11. streams                    (int64)
12. log_streams                (float64)
13. weeks_on_chart             (Int64)
14. peak_rank                  (Int64)
15. danceability               (float64)
16. energy                     (float64)
17. loudness                   (float64)
18. speechiness                (float64)
19. acousticness               (float64)
20. instrumentalness           (float64)
21. liveness                   (float64)
22. valence                    (float64)
23. tempo                   

In [14]:
num_cols = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'speechiness', 'instrumentalness', 'liveness',
            'streams', 'rank', 'weeks_on_chart']
available = [c for c in num_cols if c in df_master.columns]
df_master[available].describe().round(3)

,danceability,energy,valence,tempo,loudness,acousticness,speechiness,instrumentalness,liveness,streams,rank,weeks_on_chart
count,1077980.000,1077980.000,1077980.000,1077980.000,1077980.000,1077980.000,1077980.000,1077980.000,1077980.000,1.078125e+06,1078125.0,1078125.0
mean,0.681,0.642,0.536,121.962,-6.360,0.271,0.106,0.012,0.168,4.681387e+05,100.501,37.414
std,0.138,0.164,0.227,29.822,2.464,0.253,0.097,0.077,0.125,1.800720e+06,57.733,45.687
min,0.086,0.003,0.000,31.262,-41.410,0.000,0.022,0.000,0.013,2.067000e+03,1.0,1.0
25%,0.591,0.535,0.367,96.991,-7.693,0.056,0.041,0.000,0.093,2.389100e+04,51.0,7.0
50%,0.701,0.661,0.539,119.986,-6.023,0.190,0.062,0.000,0.118,5.985100e+04,101.0,20.0
75%,0.787,0.766,0.710,141.020,-4.601,0.422,0.132,0.000,0.203,2.718270e+05,150.0,49.0
max,0.986,1.000,0.992,232.018,1.906,0.995,0.952,0.990,0.990,8.495293e+07,200.0,290.0


In [15]:
print('Top 10 quốc gia có nhiều dòng xếp hạng nhất:')
print(df_master['country'].value_counts().head(10))

if 'income_group' in df_master.columns:
    print('\nPhân phối theo nhóm thu nhập:')
    print(df_master['income_group'].value_counts(dropna=False))

if 'region_wb' in df_master.columns:
    print('\nPhân phối theo khu vực:')
    print(df_master['region_wb'].value_counts(dropna=False))

Top 10 quốc gia có nhiều dòng xếp hạng nhất:
country
Global            26598
Australia         15200
Austria           15200
Belgium           15200
Brazil            15200
Bulgaria          15200
Canada            15200
Cyprus            15200
Czech Republic    15200
Denmark           15200
Name: count, dtype: int64

Phân phối theo nhóm thu nhập:
income_group
High income            637962
Upper middle income    270580
Lower middle income    142985
NaN                     26598
Name: count, dtype: int64

Phân phối theo khu vực:
region_wb
Europe & Central Asia         464566
Latin America & Caribbean     261768
East Asia & Pacific           182200
Middle East & North Africa     75793
North America                  30400
NaN                            26598
Sub-Saharan Africa             19600
South Asia                     17200
Name: count, dtype: int64


## 8. Xuất master dataset và dataset cho Machine Learning

- data/processed/merged_spotify_data.csv: file master đầy đủ 42 cột (~338.6 MB).
- data/processed/tracks_ml_data.csv: mỗi bài một dòng cấp độ song-level cho phần ML (~5.0 MB).


In [16]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
# 1. file master đầy đủ
master_path = os.path.join(PROCESSED_DIR, 'merged_spotify_data.csv')
df_master.to_csv(master_path, index=False)
mb = os.path.getsize(master_path) / 1e6
print(f'Master: {master_path} ({len(df_master):,} dòng x {len(df_master.columns)} cột, {mb:.1f} MB)')
# 2. dữ liệu cấp bài hát cho ML (tracks_ml_data.csv)
ml_data = generate_ml_dataset(df_master, save=True)


Master: D:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics\data\processed\merged_spotify_data.csv (1,078,125 dòng x 42 cột, 338.6 MB)

Tạo dữ liệu cấp bài hát cho ML


  D:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics\data\processed\tracks_ml_data.csv (36,699 bài, 18 cột, 5.0 MB, tỷ lệ hit 35.9%)


## Tổng kết

Đã hoàn thành các phần sau:

| Nội dung | Kết quả | Vị trí trong notebook |
|---|---|---|
| Nạp dữ liệu | 5 bảng, khoảng 1.9 triệu bản ghi | Mục 1, 1b |
| Làm sạch | Xử lý missing, bỏ trùng, kẹp giá trị ngoài miền, chuẩn hóa tên 7 quốc gia World Bank, bỏ dòng streams <= 0 | Mục 2, 3, 4 |
| Biến đổi và join | 4 phép LEFT JOIN theo `country_code`, 13 cột tính thêm, master 42 cột | Mục 5, 6, 8 |
| Khám phá dữ liệu (EDA) | Chuyển sang thực hiện toàn diện tại `02_eda.ipynb` | `notebooks/02_eda.ipynb` |

Các file đầu ra:

- `data/processed/merged_spotify_data.csv`: 1,078,125 dòng x 42 cột (~338.6 MB)
- `data/processed/tracks_ml_data.csv`: 36,699 bài x 18 cột (~5.0 MB)